In [1]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check GPU status
!nvidia-smi

Wed Jul 15 19:14:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libraries

In [3]:
%%capture
!uv pip uninstall torchao torchaudio torchvision -y
!uv pip install \
  "transformers==4.53.3" \
  "peft==0.17.1" \
  "trl" \
  "accelerate" \
  "bitsandbytes" \
  "wandb"

In [4]:
import os
import math
import json
import torch
import torch.nn as nn
import wandb
import logging
from datetime import datetime
from transformers import (
    AutoModelForMaskedLM, 
    AutoModelForQuestionAnswering,
    DataCollatorForLanguageModeling, 
    DataCollatorWithPadding,
    AutoTokenizer,
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
    TrainerState,
)
from huggingface_hub import snapshot_download
from peft import LoraConfig
from safetensors.torch import load_file, save_file
from datasets import load_dataset, Dataset

# Configurations

In [5]:
# Run configuration
SEED = 42
USERNAME = 'alxxtexxr'
LANG = 'vi'  # e.g., 'en' | 'ja' | 'id'
TASK = 'wikipedia'  # 'wikipedia' | 'squad'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Model configuration
MODEL_ID = 'FacebookAI/xlm-roberta-base'
MODEL_NAME = 'XLM-R-Base'

# LegameX configuration
# ---- Reference LoRA ----
REF_LORA_ID = 'alxxtexxr/XLM-R-Base-wikipedia-en-5K-LoRA-v260715080413'
REF_LORA_CKPT_STEP = 60

# ---- Transfer LoRA ----
# TFR_LORA_RANK = 16
# TFR_LORA_ALPHA = 16
# TFR_LORA_DROPOUT = 0.0

# ---- Gate ----
GATE_RANK = 1
GATE_ALPHA = 1
GATE_WARMUP_STEPS = 0

# Data configuration
TRAIN_SIZE = 5000
VAL_SIZE = 625

# Training configuration
MINI_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
NUM_EPOCHS = 20
WARMUP_STEPS = 50
LR = 2e-4
MLM_PROB = 0.15

In [6]:
# # Resume training configuration
# resume_from_checkpoint = bool(RESUME_MODEL_ID)
# if resume_from_checkpoint:
#     model_name = RESUME_MODEL_ID
#     run_name = model_name.split('/')[-1]
#     hub_model_id = RESUME_MODEL_ID
    
#     from huggingface_hub import snapshot_download
#     snapshot_download(repo_id=hub_model_id, local_dir=model_name)
    
#     if RESUME_CKPT_STEP:
#         resume_from_checkpoint = f"{hub_model_id}/checkpoint-{RESUME_CKPT_STEP}"
#         # Ensure the checkpoint exists
#         assert os.path.exists(resume_from_checkpoint), f"Checkpoint {resume_from_checkpoint} does not exist."
# else:
run_name = f'{MODEL_NAME}-{TASK}-{LANG}-{TRAIN_SIZE/1000:g}K-LegameX-LoRA-v{datetime.now().strftime("%y%m%d%H%M%S")}'
hub_model_id = f'{USERNAME}/{run_name}'
base_hub_model_id, version = hub_model_id.rsplit('-v', 1)
hub_merged_model_id = f'{base_hub_model_id}-Merged-v{version}'

# print("Resume from checkpoint:", resume_from_checkpoint)
print("Model name:", MODEL_NAME)
print("Run name:", run_name)
print("Hub model ID:", hub_model_id)
print("Hub merged model ID:", hub_merged_model_id)

Model name: XLM-R-Base
Run name: XLM-R-Base-wikipedia-vi-5K-LegameX-LoRA-v260715191442
Hub model ID: alxxtexxr/XLM-R-Base-wikipedia-vi-5K-LegameX-LoRA-v260715191442
Hub merged model ID: alxxtexxr/XLM-R-Base-wikipedia-vi-5K-LegameX-LoRA-Merged-v260715191442


In [7]:
# Set environment variables for wandb logging
os.environ['WANDB_PROJECT'] = 'legamex'
os.environ['WANDB_NAME'] = run_name
# os.environ['WANDB_LOG_MODEL'] = 'checkpoint' # Control whether checkpoints get uploaded to wandb as artifacts

# Utilities

In [8]:
# LoRA utilities
def download_hf_model(
        repo_id, 
        ckpt_step, 
        max_checkpoints=10_000,
        ckpt_interval=25,
    ):
    local_dir = repo_id.split('/')[-1]
    ignore_checkpoints = None
    
    if ckpt_step is not None:
        ignore_checkpoints = [f'checkpoint-{i}/*' for i in range(0, max_checkpoints, ckpt_interval) if i != ckpt_step]

    snapshot_download(
        repo_id=repo_id,
        local_dir=local_dir,
        ignore_patterns=ignore_checkpoints,
    )

    ckpt_dir = None
    if ckpt_step is not None:
        ckpt_dir = os.path.join(local_dir, f'checkpoint-{ckpt_step}')
    return local_dir, ckpt_dir

def check_parameter(n, p):
    print(f"{'name':<8}:", n)
    print(f"{'device':<8}:", p.device)
    print(f"{'dtype':<8}:", p.dtype)
    print(f"{'mean':<8}:", p.mean().item())
    print(f"{'min':<8}:", p.min().item())
    print(f"{'max':<8}:", p.max().item())

def check_weights(model, infix, limit=1):
    for i, (n, p) in enumerate(model.named_parameters()):
        if infix in n:
            check_parameter(n, p)
            print()
            if i >= limit - 1:
                break

# Data utilities
def load_train_val_datasets(
    lang, # e.g., 'en' | 'ja' | 'id'
    task, # 'wikipedia' | 'squad'
    train_size, val_size,
    token=None,
):
    # Validate that if the task is 'squad', the language must be 'en'
    if task == 'squad':
        assert lang == 'en', "SQuAD is English-only."
    
    # Define dataset configurations for each task
    data_configs = {
        'wikipedia': {
            'data_id': 'wikimedia/wikipedia',
            'data_dir': f'20231101.{lang}',
            'train_split': 'train',
            'val_split': 'train',
        },
        'squad': {
            'data_id': 'rajpurkar/squad',
            'data_dir': None,
            'train_split': 'train',
            'val_split': 'validation',
        },
    }
    
    # Validate that the specified task is supported
    assert task in data_configs, (
        f"Unsupported task: {task}. "
        f"Supported tasks: {list(data_configs.keys())}"
    )

    # Set up Hugging Face dataset configuration
    data_id = data_configs[task]['data_id']
    data_dir = data_configs[task]['data_dir']
    train_split = data_configs[task]['train_split']
    val_split = data_configs[task]['val_split']

    if train_split == val_split:
        # If the train and validation splits are the same, we need to sample from the same dataset stream
        dataset_stream = load_dataset(
            data_id,
            data_dir=data_dir,
            split=train_split,
            streaming=True,
            token=token,
        )

        train_data = []
        val_data = []

        for i, example in enumerate(dataset_stream):
            if i < train_size:
                train_data.append(example)
            elif i < train_size + val_size:
                val_data.append(example)
            else:
                break

    else:
        # If the train and validation splits are different, we can sample from each split separately
        def sample_split(split, size):
            dataset_stream = load_dataset(
                data_id,
                data_dir=data_dir,
                split=split,
                streaming=True,
            )

            data = []
            for i, example in enumerate(dataset_stream):
                if i >= size:
                    break
                data.append(example)
            return data

        train_data = sample_split(train_split, train_size)
        val_data = sample_split(val_split, val_size)

    return (
        Dataset.from_list(train_data),
        Dataset.from_list(val_data),
    )

# Model

In [10]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Determine the appropriate model classes based on the task
if TASK == 'squad':
    model_cls = AutoModelForQuestionAnswering
    task_type = 'QUESTION_ANS'
    data_collator = DataCollatorWithPadding(tokenizer)
    label_names = ['start_positions', 'end_positions']
else:
    model_cls = AutoModelForMaskedLM
    task_type = 'FEATURE_EXTRACTION'
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=MLM_PROB,
    )
    label_names = ['labels']

# Load the base model
base_model = model_cls.from_pretrained(MODEL_ID)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
Some weights of the model checkpoint at FacebookAI/xlm-roberta-base were not used when initializing XLMRobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model tha

In [11]:
class LegamexLora(nn.Module):
    def __init__(self, base_module, ref_lora_config, tfr_lora_config, 
                 gate_warmup_steps, gate_scheduler_type='cosine', 
                 gate_rank=1, gate_alpha=1, gate_dropout=0.0, 
                 gate_bias=False, gate_use_rslora=False):
        super().__init__()
        
        assert gate_scheduler_type in ['linear', 'cosine', 'quadratic', 'cubic', 'sqrt'], f"Unknown gate scheduler type: {gate_scheduler_type}"
        
        self.base_module = base_module
        self.gate_warmup_steps = gate_warmup_steps
        self.gate_scheduler_type = gate_scheduler_type
        self.training_step = 0
        
        # Determine in_features and out_features from the base module
        in_features = getattr(base_module, 'in_features', None)
        out_features = getattr(base_module, 'out_features', None)
        if in_features is None or out_features is None:
            raise ValueError(f"Cannot determine in_features or out_features from {base_module}.")
        
        # Initialize reference and transfer LoRA modules
        self.lora = nn.ModuleDict({
            'ref': self._init_lora_module(in_features, out_features, 
                                          rank=ref_lora_config.r, 
                                          alpha=ref_lora_config.lora_alpha, 
                                          dropout=ref_lora_config.lora_dropout, 
                                          bias=False if ref_lora_config.bias == 'none' else True, 
                                          use_rslora=ref_lora_config.use_rslora),
            'tfr': self._init_lora_module(in_features, out_features, 
                                          rank=tfr_lora_config.r, 
                                          alpha=tfr_lora_config.lora_alpha, 
                                          dropout=tfr_lora_config.lora_dropout, 
                                          bias=False if tfr_lora_config.bias == 'none' else True, 
                                          use_rslora=tfr_lora_config.use_rslora),
        })
        
        # Initialize the modules for the gate
        self.gate = self._init_lora_module(in_features, out_features, 
                                           rank=gate_rank, alpha=gate_alpha, 
                                           dropout=gate_dropout, bias=gate_bias, 
                                           use_rslora=gate_use_rslora)
        self.gate_act = nn.Sigmoid()
        
    def _init_lora_module(self, in_features, out_features, rank, alpha, dropout, bias, use_rslora):
        device = self.base_module.weight.device
        
        # Initialize the LoRA module with the extracted parameters
        scaling = alpha / math.sqrt(rank) if use_rslora else alpha / rank
        dropout_module = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        lora_module = nn.ModuleDict({
            'A': nn.Linear(in_features, rank, bias=bias, device=device),
            'B': nn.Linear(rank, out_features, bias=bias, device=device),
            'dropout': dropout_module,
        })
        lora_module.scaling = scaling
        lora_module.use_bias = bias
        
        # Initialize weights for LoRA modules
        # nn.init.normal_(lora_module.A.weight, mean=0.0, std=1 / math.sqrt(rank))
        nn.init.kaiming_uniform_(lora_module.A.weight, a=math.sqrt(5))
        nn.init.zeros_(lora_module.B.weight)
        
        return lora_module

    def forward(self, x):
        # Compute the base layer output
        base_out = self.base_module(x)
        
        # Check if the input tensor needs to be converted to the LoRA module's dtype
        requires_conversion = not torch.is_autocast_enabled()
        if requires_conversion:
            # TODO: Add assertion to check if all weights have the same dtype
            x = x.to(self.lora.tfr.A.weight.dtype)
        
        # Compute the outputs for the reference and transfer LoRA
        ref_lora_out = self.lora.ref.B(self.lora.ref.dropout(self.lora.ref.A(x))) * self.lora.ref.scaling
        tfr_lora_out = self.lora.tfr.B(self.lora.tfr.dropout(self.lora.tfr.A(x))) * self.lora.tfr.scaling
        
        # Compute the gate output and its complement
        gate_out = self.gate_act(self.gate.B(self.gate.dropout(self.gate.A(x))))
        gate_out = gate_out * self.gate_scheduler() # Schedule the gate output based on the training step and warmup steps
        gate_comp_out = torch.ones(gate_out.shape, dtype=gate_out.dtype, device=gate_out.device) - gate_out
        
        # Compute the LoRA output
        lora_out = gate_out * ref_lora_out + gate_comp_out * tfr_lora_out
        
        # Check if the LoRA output needs to be converted back to the base layer's dtype
        if requires_conversion:
            lora_out = lora_out.to(base_out.dtype)
            
        return base_out + lora_out

    def gate_scheduler(self):
        start = 0.0
        end = 1.0
        step = self.training_step
        warmup_steps = self.gate_warmup_steps
        scheduler_type = self.gate_scheduler_type
        
        if warmup_steps <= 0:
            return end

        t = min(max(step / warmup_steps, 0.0), 1.0)

        if scheduler_type == 'linear':
            p = t
        elif scheduler_type == 'cosine':
            # cosine warmup
            p = 0.5 * (1.0 - math.cos(math.pi * t))
        elif scheduler_type == 'quadratic':
            p = t**2
        elif scheduler_type == 'cubic':
            p = t**3
        elif scheduler_type == 'sqrt':
            p = math.sqrt(t)
        else:
            raise ValueError(f"Unknown gate scheduler type: {scheduler_type}")

        return start + (end - start) * p
    
    def set_training_step(self, training_step):
        self.training_step = training_step
    
    def load_lora_weights(self, state_dict, prefix, lora_type):
        assert lora_type in ['ref', 'tfr'], "lora_type must be either 'ref' or 'tfr'."
        
        device_A = self.lora[lora_type].A.weight.device
        device_B = self.lora[lora_type].B.weight.device

        self.lora[lora_type].A.weight.data = state_dict[f'{prefix}.lora_A.weight'].to(device_A)
        self.lora[lora_type].B.weight.data = state_dict[f'{prefix}.lora_B.weight'].to(device_B)

        if self.lora[lora_type].use_bias:
            if f'{prefix}.lora_A.bias' in state_dict:
                self.lora[lora_type].A.bias.data = state_dict[f'{prefix}.lora_A.bias'].to(device_A)
            if f'{prefix}.lora_B.bias' in state_dict:
                self.lora[lora_type].B.bias.data = state_dict[f'{prefix}.lora_B.bias'].to(device_B)

class LegamexLoraModel(nn.Module):
    def __init__(self, base_model, task_type, ref_lora_config, tfr_lora_config, 
                 gate_warmup_steps, gate_scheduler_type='cosine', 
                 gate_rank=1, gate_alpha=1, gate_dropout=0.0, 
                 gate_bias=False, gate_use_rslora=False):
        
        super().__init__()
        
        self.base_model_name_or_path = (
            ref_lora_config.base_model_name_or_path
            if hasattr(ref_lora_config, 'base_model_name_or_path')
            else base_model.config.name_or_path
        )
        if base_model.config.name_or_path != self.base_model_name_or_path:
            print(f"Note: using base model name from reference LoRA: {self.base_model_name_or_path}")
                
        self.base_model = base_model
        self.task_type = task_type
        self.ref_lora_config = ref_lora_config
        self.tfr_lora_config = tfr_lora_config
        self.gate_warmup_steps = gate_warmup_steps
        self.gate_scheduler_type = gate_scheduler_type
        self.gate_rank = gate_rank
        self.gate_alpha = gate_alpha
        self.gate_dropout = gate_dropout
        self.gate_bias = gate_bias
        self.gate_use_rslora = gate_use_rslora

        # Wrap target modules with LegameX
        self.legamex_modules = {}
        self._wrap_target_modules_with_legamex()

    def _wrap_target_modules_with_legamex(self):
        assert self.ref_lora_config.target_modules == self.tfr_lora_config.target_modules, "Reference and Transfer LoRA must have the same target modules."
        target_modules = self.tfr_lora_config.target_modules
        
        for module_name, module in self.base_model.named_modules():
            # Skip the module if it's already wrapped with the LegamexLora module
            if isinstance(module, LegamexLora):
                module_name = module_name.rsplit('model.', 1)[-1]
                module_name = module_name.replace('.', '__DOT__')
                self.legamex_modules[module_name] = module
                continue
            
            if any(target_module in module_name for target_module in target_modules):
                # Wrap the module with the LegamexLora module
                parent_module, child_name = self._get_parent_module(module_name)
                legamex_module = LegamexLora(
                    base_module=module,
                    ref_lora_config=self.ref_lora_config,
                    tfr_lora_config=self.tfr_lora_config,
                    gate_warmup_steps=self.gate_warmup_steps,
                    gate_scheduler_type=self.gate_scheduler_type,
                    gate_rank=self.gate_rank,
                    gate_alpha=self.gate_alpha,
                    gate_dropout=self.gate_dropout,
                    gate_bias=self.gate_bias,
                    gate_use_rslora=self.gate_use_rslora,
                )
                setattr(parent_module, child_name, legamex_module)
                
                # Store the LegamexLora module for later weight loading
                module_name = module_name.rsplit('model.', 1)[-1]
                module_name = module_name.replace('.', '__DOT__')
                self.legamex_modules[module_name] = legamex_module
                
            # Store the module if it's the qa_outputs module and task_type='QUESTION_ANS'
            if isinstance(module, nn.Linear) and 'qa_outputs' in module_name and self.task_type == 'QUESTION_ANS':
                module_name = module_name.rsplit('model.', 1)[-1]
                module_name = module_name.replace('.', '__DOT__')
                self.legamex_modules[module_name] = module
                
        # Freeze all modules, then unfreeze all LegameX-specific modules except reference LoRA modules
        self.freeze_all(verbose=False)
        self.unfreeze_legamex_except_ref_lora(verbose=True)
    
    def _get_parent_module(self, module_name):
        parts = module_name.split('.')
        parent_module = self.base_model
        for part in parts[:-1]:
            parent_module = getattr(parent_module, part)
        return parent_module, parts[-1]
    
    def freeze_all(self, verbose=False):
        for p in self.base_model.parameters():
            p.requires_grad = False
        if verbose:
            print("All modules are frozen.")

    def unfreeze_legamex_except_ref_lora(self, verbose=False):
        for module_name, module in self.legamex_modules.items():
            if isinstance(module, LegamexLora):
                # Unfreeze the transfer LoRA A and B weights
                module.lora.tfr.A.weight.requires_grad = True
                module.lora.tfr.B.weight.requires_grad = True
                if module.lora.tfr.use_bias:
                    if hasattr(module.lora.tfr.A, 'bias') and module.lora.tfr.A.bias is not None:
                        module.lora.tfr.A.bias.requires_grad = True
                    if hasattr(module.lora.tfr.B, 'bias') and module.lora.tfr.B.bias is not None:
                        module.lora.tfr.B.bias.requires_grad = True
                
                # Unfreeze the gate weights
                module.gate.A.weight.requires_grad = True
                module.gate.B.weight.requires_grad = True
                if module.gate.use_bias:
                    if hasattr(module.gate.A, 'bias') and module.gate.A.bias is not None:
                        module.gate.A.bias.requires_grad = True
                    if hasattr(module.gate.B, 'bias') and module.gate.B.bias is not None:
                        module.gate.B.bias.requires_grad = True
                        
            elif isinstance(module, nn.Linear) and 'qa_outputs' in module_name and self.task_type == 'QUESTION_ANS':
                # Unfreeze the qa_outputs weight and bias if task_type='QUESTION_ANS'
                module.weight.requires_grad = True
                if hasattr(module, 'bias') and module.bias is not None:
                        module.bias.requires_grad = True

        if verbose:
            trainable = sum(p.numel() for p in self.base_model.parameters() if p.requires_grad)
            total = sum(p.numel() for p in self.base_model.parameters())
            print(f"Trainable parameters: {trainable:,} / {total:,}")
    
    def load_lora_weights(self, lora_path, lora_type):
        assert lora_type in ['ref', 'tfr'], "Invalid LoRA type. Must be 'ref' or 'tfr'."
        
        if not os.path.exists(lora_path):
            raise FileNotFoundError(f"LoRA weights file not found: {lora_path}")
        
        if lora_path.endswith('.safetensors'):
            state_dict = load_file(lora_path)
        else:
            state_dict = torch.load(lora_path, map_location='cpu')

        # prefix = list(state_dict.keys())[0].rsplit('model.', 1)[0] + 'model.'
        lora_A_keys = [k for k in state_dict.keys() if k.endswith('.lora_A.weight')]
        if lora_A_keys:
            # e.g., 'base_model.model.encoder.layer.0.attention.self.query.lora_A.weight'
            prefix = lora_A_keys[0][:-len('.lora_A.weight')] + '.'
        else:
            prefix = ''  # or raise an error

        for lora_module_name, lora_module in self.legamex_modules.items():
            lora_module_name = lora_module_name.replace('__DOT__', '.')
            lora_module_name = prefix + lora_module_name
            if f'{lora_module_name}.lora_A.weight' in state_dict and f'{lora_module_name}.lora_B.weight' in state_dict:
                lora_module.load_lora_weights(state_dict, lora_module_name, lora_type)
            else:
                # TODO: Print warning message
                pass

        print(f"LoRA weights loaded from {lora_path} into the model.")

    def forward(self, *args, **kwargs):
        # Remove Trainer-only kwargs before forwarding to the base model
        kwargs.pop('num_items_in_batch', None)
        return self.base_model.forward(*args, **kwargs)
    
    def _json_safe(self, obj):
        """Recursively convert non-JSON-serializable types (e.g. sets) to JSON-safe ones."""
        if isinstance(obj, set):
            return list(obj)
        if isinstance(obj, dict):
            return {k: self._json_safe(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [self._json_safe(v) for v in obj]
        return obj
    
    def save_pretrained(self, save_dir):
        """Save base model + LegameX adapters in a Hugging Face-friendly way."""
        
        tfr_lora_dir = os.path.join(save_dir, 'tfr_lora_gated')
        
        os.makedirs(save_dir, exist_ok=True)
        os.makedirs(tfr_lora_dir, exist_ok=True)
        
        state_dict = {}
        tfr_lora_state_dict = {}
        for module_name, module in self.legamex_modules.items():
            for param_name, param in module.named_parameters():
                full_module_name = f"{module_name}.{param_name}"
                state_dict[full_module_name] = param.detach().cpu()
                
                # Store transfer LoRA and qa_outputs parameters separately
                if 'tfr' in param_name or 'qa_outputs' in param_name:
                    tfr_lora_state_dict[full_module_name] = param.detach().cpu()
                
        torch.save(state_dict, os.path.join(save_dir, 'legamex.pt'))
        
        # Save transfer LoRA and its configuration in a separate directory for easier loading
        save_file(tfr_lora_state_dict, os.path.join(tfr_lora_dir, 'adapter_model.safetensors'))
        with open(os.path.join(tfr_lora_dir, 'adapter_config.json'), 'w') as f:
            json.dump(self._json_safe(self.tfr_lora_config.to_dict()), f)
        
        # Save a small config so we can rebuild the wrappers
        config = {
            'base_model_name_or_path': self.base_model_name_or_path,
            'task_type': self.task_type,
            'ref_lora_config': self._json_safe(self.ref_lora_config.to_dict()),
            'tfr_lora_config': self._json_safe(self.tfr_lora_config.to_dict()),
            'gate_warmup_steps': self.gate_warmup_steps,
            'gate_scheduler_type': self.gate_scheduler_type,
            'gate_rank': self.gate_rank,
            'gate_alpha': self.gate_alpha,
            'gate_dropout': self.gate_dropout,
            'gate_bias': self.gate_bias,
            'gate_use_rslora': self.gate_use_rslora,
        }
        with open(os.path.join(save_dir, 'legamex_config.json'), 'w') as f:
            json.dump(config, f)

    @classmethod
    def from_pretrained(cls, save_dir, **kwargs):
        """Load a LegamexLoraModel from a directory saved with save_pretrained."""
        
        # Load the custom config
        with open(os.path.join(save_dir, 'legamex_config.json'), 'r') as f:
            config = json.load(f)
            
        # Re-create base model
        base_model_id = config['base_model_name_or_path']
        task_type = config['task_type']
        if task_type == 'QUESTION_ANS':
            base_model = AutoModelForQuestionAnswering.from_pretrained(base_model_id)
        elif task_type == 'FEATURE_EXTRACTION':
            base_model = AutoModelForMaskedLM.from_pretrained(base_model_id)
        else:
            raise ValueError(f"Unknown task type: {task_type}")
        
        ref_lora_config = LoraConfig(**config['ref_lora_config'])
        tfr_lora_config = LoraConfig(**config['tfr_lora_config'])
        
        # Instantiate LegamexLoraModel (wraps the base model)
        model = cls(
            base_model=base_model,
            task_type=task_type,
            ref_lora_config=ref_lora_config,
            tfr_lora_config=tfr_lora_config,
            gate_warmup_steps=config['gate_warmup_steps'],
            gate_scheduler_type=config['gate_scheduler_type'],
            gate_rank=config['gate_rank'],
            gate_alpha=config['gate_alpha'],
            gate_dropout=config['gate_dropout'],
            gate_bias=config['gate_bias'],
            gate_use_rslora=config['gate_use_rslora'],
        )
        
        full_state_dict = torch.load(os.path.join(save_dir, 'legamex.pt'), map_location='cpu')

        for key, tensor in full_state_dict.items():
            parts = key.split('.', 1)   # 'module_name' + 'rest.of.path'
            if len(parts) == 2:
                module_name, param_name = parts
                module = model.legamex_modules.get(module_name)
                if module is not None:
                    # Look up the parameter inside the module using a flat dict
                    param = dict(module.named_parameters()).get(param_name)
                    if param is not None:
                        param.data.copy_(tensor)
        
        return model
        
    def set_training_step(self, training_step):
        for module in self.legamex_modules.values():
            if hasattr(module, 'set_training_step'):    # skip Linear layers
                module.set_training_step(training_step)
    
    def __getattr__(self, name):
        try:
            return super().__getattr__(name) # Try getting attribute from self
        except AttributeError:
            return getattr(self.base_model, name) # Fallback to base_model
    
# Load reference LoRA configuration
if len(REF_LORA_ID.split('/')) == 2:
    _, ref_lora_dir = download_hf_model(REF_LORA_ID, REF_LORA_CKPT_STEP)
ref_lora_path = os.path.join(ref_lora_dir, 'adapter_model.safetensors')
ref_lora_config = LoraConfig.from_pretrained(ref_lora_dir)

# Initialize the LegameX model
legamex_model = LegamexLoraModel(
    base_model,
    task_type=task_type,
    ref_lora_config=ref_lora_config,
    tfr_lora_config=ref_lora_config, # Set to be the same as reference LoRA
    gate_warmup_steps=GATE_WARMUP_STEPS,
)
legamex_model.to(DEVICE)

Fetching 65 files:   0%|          | 0/65 [00:00<?, ?it/s]

Trainable parameters: 2,820,096 / 283,769,490


LegamexLoraModel(
  (base_model): XLMRobertaForMaskedLM(
    (roberta): XLMRobertaModel(
      (embeddings): XLMRobertaEmbeddings(
        (word_embeddings): Embedding(250002, 768, padding_idx=1)
        (position_embeddings): Embedding(514, 768, padding_idx=1)
        (token_type_embeddings): Embedding(1, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): XLMRobertaEncoder(
        (layer): ModuleList(
          (0-11): 12 x XLMRobertaLayer(
            (attention): XLMRobertaAttention(
              (self): XLMRobertaSdpaSelfAttention(
                (query): LegamexLora(
                  (base_module): Linear(in_features=768, out_features=768, bias=True)
                  (lora): ModuleDict(
                    (ref): ModuleDict(
                      (A): Linear(in_features=768, out_features=16, bias=False)
                      (B): Linear(in_features=16, out_features=768, bias

In [12]:
print("Check unloaded LegameX-specific weights:")
check_weights(legamex_model, infix='lora.ref')
check_weights(legamex_model, infix='lora.tfr')
check_weights(legamex_model, infix='gate')

legamex_model.load_lora_weights(ref_lora_path, lora_type='ref')
print()

print("Check loaded LegameX-specific weights:")
check_weights(legamex_model, infix='lora.ref')
check_weights(legamex_model, infix='lora.tfr')
check_weights(legamex_model, infix='gate')

Check unloaded LegameX-specific weights:
name    : base_model.roberta.encoder.layer.0.attention.self.query.lora.ref.A.weight
device  : cuda:0
dtype   : torch.float32
mean    : 0.00015419127885252237
min     : -0.03607944771647453
max     : 0.03607633337378502

name    : base_model.roberta.encoder.layer.0.attention.self.query.lora.tfr.A.weight
device  : cuda:0
dtype   : torch.float32
mean    : -0.00012930374941788614
min     : -0.0360826700925827
max     : 0.036075714975595474

name    : base_model.roberta.encoder.layer.0.attention.self.query.gate.A.weight
device  : cuda:0
dtype   : torch.float32
mean    : 0.0002927693712990731
min     : -0.035936206579208374
max     : 0.035735730081796646

LoRA weights loaded from XLM-R-Base-wikipedia-en-5K-LoRA-v260715080413/checkpoint-60/adapter_model.safetensors into the model.

Check loaded LegameX-specific weights:
name    : base_model.roberta.encoder.layer.0.attention.self.query.lora.ref.A.weight
device  : cuda:0
dtype   : torch.float32
mean    :

In [13]:
# Sanity check: ensure that the only trainable parameters are the transfer LoRA, gate, and qa_outputs weights and biases
# for n, p in legamex_model.named_parameters():
#     print(n, p.requires_grad)

# Data

In [14]:
# Load the dataset
train_dataset, val_dataset = load_train_val_datasets(
    lang=LANG, task=TASK,
    train_size=TRAIN_SIZE, val_size=VAL_SIZE
)

print("Train dataset:")
print(train_dataset)
print()
print("Validation dataset:")
print(val_dataset)

Train dataset:
Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 5000
})

Validation dataset:
Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 625
})


In [15]:
# Preprocess the dataset
if TASK == 'squad':
    def preprocess_squad(examples):
        # Tokenize question + context with offset mapping
        tokenized = tokenizer(
            examples['question'],
            examples['context'],
            truncation='only_second',
            max_length=384,
            # stride=128,
            return_offsets_mapping=True,
            # padding='max_length',
        )

        # Prepare label lists
        start_positions = []
        end_positions = []

        for i, offsets in enumerate(tokenized['offset_mapping']):
            # Find which tokens belong to the context (not the question, not special tokens)
            sequence_ids = tokenized.sequence_ids(i)

            # SQuAD v1.1 always has exactly one answer; take the first
            answer = examples['answers'][i]
            answer_start_char = answer['answer_start'][0]
            answer_text = answer['text'][0]
            answer_end_char = answer_start_char + len(answer_text)

            # Locate the token span that corresponds to the answer
            token_start = None
            token_end = None
            for idx, (offset_start, offset_end) in enumerate(offsets):
                # Ignore question tokens and special tokens
                if sequence_ids[idx] != 1:
                    continue
                # Token fully inside answer
                if offset_start >= answer_start_char and offset_end <= answer_end_char:
                    if token_start is None:
                        token_start = idx
                    token_end = idx
                # Token partially overlapping (shouldn’t happen with whitespace splits)
                elif offset_start < answer_end_char and offset_end > answer_start_char:
                    if token_start is None:
                        token_start = idx
                    token_end = idx

            # If answer is out of bounds (truncated), set to CLS token index
            if token_start is None or token_end is None:
                token_start = 0
                token_end = 0

            start_positions.append(token_start)
            end_positions.append(token_end)

        tokenized['start_positions'] = start_positions
        tokenized['end_positions'] = end_positions
        
        del tokenized['offset_mapping']

        return tokenized
    
    train_dataset = train_dataset.map(preprocess_squad, batched=True, remove_columns=train_dataset.column_names)
    val_dataset = val_dataset.map(preprocess_squad, batched=True, remove_columns=val_dataset.column_names)
else:
    def tokenize_text(examples):
        tokenized = tokenizer(
            examples['text'],
            max_length=512,
            truncation=True,
            padding='max_length',
        )
        tokenized['labels'] = tokenized['input_ids'].copy()
        return tokenized

    train_dataset = train_dataset.map(tokenize_text, batched=True, remove_columns=train_dataset.column_names)
    val_dataset = val_dataset.map(tokenize_text, batched=True, remove_columns=val_dataset.column_names)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/625 [00:00<?, ? examples/s]

# Training

In [16]:
logger = logging.getLogger(__name__)

class LegamexTrainer(Trainer):
    def _save(self, output_dir=None, state_dict=None):
        output_dir = output_dir if output_dir is not None else self.args.output_dir
        os.makedirs(output_dir, exist_ok=True)

        # Use our own lightweight saver instead of Trainer's default full state_dict dump
        self.model.save_pretrained(output_dir)

        # Still save processing_class/tokenizer + training args like Trainer normally would
        if self.processing_class is not None:
            self.processing_class.save_pretrained(output_dir)
        torch.save(self.args, os.path.join(output_dir, 'training_args.bin'))
        
    def _load_from_checkpoint(self, resume_from_checkpoint, model=None):
        config_file = os.path.join(resume_from_checkpoint, 'legamex_config.json')
        if not os.path.isfile(config_file):
            # Fall back to original behaviour for regular checkpoints
            super()._load_from_checkpoint(resume_from_checkpoint, model=model)
            return

        # Reconstruct model using our lightweight adapter files
        model = LegamexLoraModel.from_pretrained(resume_from_checkpoint)
        model.to(self.args.device)
        self.model = model
        self.model_wrapped = model

        # Restore optimizer & scheduler
        device = next(self.model.parameters()).device
        optimizer_path = os.path.join(resume_from_checkpoint, 'optimizer.pt')
        if os.path.isfile(optimizer_path):
            self.optimizer.load_state_dict(
                torch.load(optimizer_path, map_location=device)
            )
        scheduler_path = os.path.join(resume_from_checkpoint, 'scheduler.pt')
        if os.path.isfile(scheduler_path):
            self.lr_scheduler.load_state_dict(
                torch.load(scheduler_path, map_location=device)
            )

        # Reload trainer state (epoch, best metrics, etc.)
        trainer_state_path = os.path.join(resume_from_checkpoint, 'trainer_state.json')
        if os.path.isfile(trainer_state_path):
            with open(trainer_state_path, 'r') as f:
                self.state = TrainerState.load_from_json(f.read())

        # Sync LegameX training step
        if hasattr(self.model, 'set_training_step'):
            self.model.set_training_step(self.state.global_step)

        logger.info(f"Resumed LegameX model from {resume_from_checkpoint}")

class GateStepCallback(TrainerCallback):
    """Updates the LegameX training step at every optimisation step."""
    def on_step_end(self, args, state, control, **kwargs):
        model = kwargs['model']
        if hasattr(model, 'set_training_step'):
            model.set_training_step(state.global_step)
        return control
    
class WandbGateStatsCallback(TrainerCallback):
    """Logs gate weight statistics at every optimisation step."""
    def on_log(self, args, state, control, model=None, logs=None, **kwargs):
        # Gather only the LegamexLora modules
        lora_modules = [m for m in model.legamex_modules.values()
                        if hasattr(m, 'gate')]  # or isinstance(m, LegamexLora)
        if not lora_modules:
            return

        # Gate scheduler alpha (same for all layers)
        first_module = lora_modules[0]
        alpha = first_module.gate_scheduler()

        # Aggregate gate weight statistics
        A_mean, A_norm = 0.0, 0.0
        B_mean, B_norm = 0.0, 0.0
        n = len(lora_modules)

        for m in lora_modules:
            A_mean += m.gate.A.weight.mean().item()
            A_norm += m.gate.A.weight.norm(p=2).item()
            B_mean += m.gate.B.weight.mean().item()
            B_norm += m.gate.B.weight.norm(p=2).item()

        wandb.log({
            'gate/scheduler_alpha': alpha,
            'gate/A_mean': A_mean / n,
            'gate/A_norm': A_norm / n,
            'gate/B_mean': B_mean / n,
            'gate/B_norm': B_norm / n,
            'step': state.global_step,
        })

In [17]:
# Calculate the maximum number of training steps
max_steps = math.ceil(len(train_dataset) / (MINI_BATCH_SIZE * GRAD_ACCUM_STEPS)) * NUM_EPOCHS
print("Calculated max steps:", max_steps)

# Set up the trainer
training_args = TrainingArguments(
    remove_unused_columns=False,
    
    # Training arguments
    seed=SEED,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    per_device_train_batch_size=MINI_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    max_steps = max_steps,
    warmup_steps = WARMUP_STEPS,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    max_grad_norm=1.0,
    weight_decay=0.01,
    
    # Validation arguments
    eval_strategy='steps',
    eval_steps=20,
    
    # Logging arguments
    logging_strategy='steps',
    logging_steps=10,
    # logging_first_step=True,
    report_to=['tensorboard', 'wandb'],
    
    # Saving arguments
    save_strategy='steps',
    save_steps=20,
    # save_total_limit=5, # 1 best + 4 recent checkpoints. WARN: It doesn't work
    
    # With load_best_model_at_end=True, your save_strategy will be ignored and default to eval_strategy.
    # So you will find one checkpoint at the end of each epoch.
    # https://discuss.huggingface.co/t/trainer-not-saving-after-save-steps/5464
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better = False,

    # run_name=run_name,
    output_dir=run_name,
    hub_model_id=hub_model_id,
    push_to_hub=True,
    hub_strategy='all_checkpoints',
    hub_always_push=True,
    
    # TEMP: Do not save and upload checkpoints for testing
    # save_strategy='no', 
    # load_best_model_at_end=False,
)
trainer = LegamexTrainer(
    model=legamex_model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    args=training_args,
    # label_names=['labels'],
    callbacks = [
        EarlyStoppingCallback(
            early_stopping_patience=3,
            # early_stopping_threshold = 0.001,
        ),
        GateStepCallback(),
        WandbGateStatsCallback(),
    ],
)
trainer.label_names = label_names

Calculated max steps: 6260


In [18]:
# Start training
trainer_stats = trainer.train(
    # resume_from_checkpoint=resume_from_checkpoint
)

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: alimtegar to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss
20,6.270100,1.409532
40,6.179100,1.367914
60,6.044000,1.371301
80,6.163700,1.353976
100,6.155000,1.359108
120,6.274300,1.341848
140,6.126300,1.363025
160,6.092300,1.360396
180,6.072800,1.357286


Could not locate the best model at XLM-R-Base-wikipedia-vi-5K-LegameX-LoRA-v260715191442/checkpoint-120/pytorch_model.bin, if you are running a distributed training on multiple nodes, you should activate `--save_on_each_node`.
